# HOUSE-3 · 4 — The quarter

**The failure this notebook prevents.** Nine experiments, each with a primary KPI and
two guardrails, is twenty-seven go/no-go decisions in a year. At the conventional 5%,
with nothing working at all, one and a third of them are *go*. Every one of those
numbers is individually defensible and individually documented, and the house is wrong
once or twice a year with a straight face — and reports it to three different boards
who will never compare notes.

The other failure is the readout that was not evidence and got counted anyway: the
search test read on an assignment nobody filed. Notebook 3 blocked it. This notebook
shows that the block *travels*: the programme's error control never sees the run,
because the lifecycle refuses to hand it over.

| | reaches into |
|---|---|
| what the lifecycle hands over | `build.readouts_across`, `build.ExperimentBuilder.runs`, `io.ExperimentRun.conformance` |
| evidence on each readout | `design.Readout`, `design.evalue`, `design.from_p_value`, `design.stagewise_tail` |
| the quarter, decided together | `design.expected_false_go`, `design.program_decisions`, `design.ProgramReport`, `design.ARBITRARY_DEPENDENCE` |
| the stream, decided on arrival | `design.online_decisions`, `design.OnlineProgram`, `design.INDEPENDENT_READOUTS` |
| the audit trail | `io.Catalog.find`, `io.Catalog.entries`, `io.CrossScopeError` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from IPython.display import display

import house as H
import year as Y
from axiom.display import enable, table
from _style import (
    AQUA, BLUE, CRITICAL, GOOD, MUTED, ORANGE, VIOLET, WARNING,
    annotate, caption, compare, curve_band, density, dumbbell, figure, heat, intervals,
    lines, mark_x, mark_y, points, shade, steps,
)

enable();  # every axiom result renders itself from here on

from scipy import stats as st

worlds = {name: H.world(name, year=1) for name in H.PARTIES}
inputs = {name: Y.design_inputs(H.world(name, year=0)) for name in H.PARTIES}
catalog, programs, builders = Y.open_house()
Y.register_definitions(builders)
plans = Y.plan_year(builders, inputs)                       # notebook 2
fielded = Y.field_year(worlds, plans)                       # the year
reads = Y.read_year(worlds, builders, plans, fielded)       # notebook 3

rows = []
for name, builder in builders.items():
    for run in builder.runs(stage="read"):
        v = run.conformance()
        rows.append([name, run.experiment, v.status, len(run.deviations), (v.reason or "—")[:60]])
table(rows, headers=("party", "experiment", "conformance", "deviations", "why"), title="every run at 'read', from the catalog")

## 1 · What the lifecycle hands over

`readouts_across` collects one `Readout` per experiment from each party's builder —
the latest filed version — and only for runs whose conformance is licensed. It carries
no p-value: the evidence comes from the analysis, not from the lifecycle, and the
builder refuses to invent one. NW-03 is absent, and nothing downstream will ever count
it.

In [ ]:
from axiom.build import readouts_across

handed = readouts_across(list(builders.values()))
table([[r.key, r.party, r.metric, r.p_value, r.evalue] for r in handed],
      headers=("key", "party", "metric", "p_value", "evalue"), title="what the programme is handed — identity only")
print("absent:", sorted(set(plans) - {r.experiment for r in handed}))

## 2 · Attaching the evidence

Each experiment's primary readout gets the p-value and e-value its own analysis
produced. NW-01 was monitored with a confidence sequence, so it has a *native* e-value
— the mixture martingale, valid at every look. GX-02 stopped on a boundary, so its
p-value is the stage-wise one, not the fixed-sample one at the stopping look. The
others were read once at the end, and their e-values are calibrated from their
p-values with `from_p_value` — a weaker object, as its docstring says, and the
programme is only as anytime-valid as its weakest input.

Guardrails are decisions too. Each party watches two metrics no channel is supposed to
move, and a programme that checks them is taking three decisions per experiment, not
one.

In [ ]:
from axiom.design import Readout, confidence_sequence, evalue, from_p_value, stagewise_tail

def primary_evidence(key):
    rd = reads[key]
    if key == "NW-01":
        exp = rd.plan.experiment
        z, _, _ = Y.weekly_path(Y.kpi_seen(worlds["northwind"], rd.fielded), rd.fielded, range(1, exp.weeks + 1))
        seq = confidence_sequence(z, [k / exp.weeks for k in range(1, exp.weeks + 1)], alpha=0.05, name="NW-01")
        return seq.final.anytime_p, seq.final.running_evalue, "confidence sequence"
    if key == "GX-02":
        path, z = rd.detail["path"], rd.detail["z"]
        tail = stagewise_tail(path.rule, path.stopped_at, z[path.stopped_at], 0.0)
        p = float(min(1.0, 2 * min(tail, 1 - tail)))
        return p, from_p_value(p), "stage-wise, calibrated"
    m = rd.measurement
    p = float(2 * st.norm.sf(abs(m.estimate / m.se)))
    return p, from_p_value(p), "fixed-sample, calibrated"

book, rows = [], []
for r in handed:
    p, e, how = primary_evidence(r.experiment)
    book.append(r.model_copy(update={"p_value": p, "evalue": e}))
    rows.append([r.key, how, f"{p:.4f}", f"{e:.2f}"])
    w = worlds[r.party]
    for name, est in Y.guardrail_frame(w, reads[r.experiment].fielded).items():
        pg = float(2 * st.norm.sf(abs(est.effect / est.se)))
        book.append(r.model_copy(update={"metric": name, "p_value": pg, "evalue": from_p_value(pg)}))
        rows.append([f"{r.party}/{r.experiment}:{name}", "guardrail, calibrated", f"{pg:.4f}", f"{from_p_value(pg):.2f}"])
table(rows, headers=("readout", "evidence", "p", "e-value"), title=f"{len(book)} readouts, {sum(1 for b in book if b.metric == 'primary')} primary")

## 3 · The quarter, decided together

Four routes. `none` is what each party would do on its own. `holm` controls the chance
of *any* false go. `benjamini_hochberg` controls the false discovery rate and needs the
readouts to be independent or positively dependent, which a book of experiments
sharing markets and a season does not obviously satisfy. `e_bh` controls the same rate
and assumes **nothing** about how the readouts depend on each other — which is why
`design.anytime` exposed its e-value in the first place.

`program_decisions` will not pick. What it does is put the uncorrected count beside
the corrected one, so that choosing `none` is a choice somebody made.

In [ ]:
from axiom.design import ARBITRARY_DEPENDENCE, expected_false_go, program_decisions

print(f"{len(book)} readouts at alpha 0.05: a null book expects {expected_false_go(len(book), 0.05):.1f} go-decisions")
reports = {}
rows = []
for method in ("none", "holm", "benjamini_hochberg", "e_bh"):
    rep = program_decisions(book, alpha=0.05, method=method, period="2026")
    reports[method] = rep
    goes = [d for d in rep.decisions if d.go]
    rows.append([method, len(goes), sum(1 for d in goes if d.metric == "primary"), sum(1 for d in goes if d.metric != "primary"),
                 ", ".join(d.key.split("/")[1] for d in goes), rep.bound[:60] + "…"])
table(rows, headers=("method", "go", "primary", "guardrail", "which", "what it controls"), title="the same 24 readouts, four ways")

Read the `none` row first. Six goes, one of them a guardrail — acme's churn metric,
which no channel in this world touches. That is the one-and-a-fifth false goes a null
book expects, arriving on schedule, and it would have gone to acme's board as "the
social holdout moved churn".

Then read the `e_bh` row, and read it carefully, because it is a lesson about what
error control is not. The single readout that survives arbitrary-dependence control is
AC-04 — the autumn search holdout whose control markets went dark, whose readout
notebook 3 showed sits two standard errors above the truth. Error control corrects the
*count* of false goes on a null book; it cannot see that a readout's number is biased,
because its inputs are p-values and a biased estimate has a very good p-value. That
fact is on the run — `downgraded`, with an unverified assumption about the population
— and it is the reader's job to weigh it, not the multiplicity's.

In [ ]:
ebh = reports["e_bh"]
print(ebh.summary())
print()
print(ebh.ledger_line().statement)
print("assumption:", ARBITRARY_DEPENDENCE.name, "|", ARBITRARY_DEPENDENCE.state, "| challenged by:", ARBITRARY_DEPENDENCE.challenged_by[:60], "…")
strongest = sorted(ebh.decisions, key=lambda d: -d.evidence)[:4]
table([[d.key, f"{d.evidence:.2f}", f"{d.threshold:.2f}", str(d.go), str(d.go_uncorrected),
        reads[d.experiment].run.conformance().status] for d in strongest],
      headers=("readout", "e-value", "e-BH cut", "go", "go uncorrected", "run conformance"), title="the four strongest readouts under e-BH")

Why does e-BH keep only one where Benjamini–Hochberg keeps six? Two reasons, and only
one of them is about the method. The e-BH cut on twenty-four readouts with one go is
`n / (alpha · k)` = 480: a book that is mostly guardrails and one-off reads sets a
high bar for itself. And seven of the eight primaries carry *calibrated* e-values,
which collapse for strong evidence — their tail is polynomial where the mixture's is
Gaussian. The table below prices the second reason: what each readout's e-value would
have been had it been monitored with a confidence sequence. It roughly doubles GX-01's
and multiplies AC-04's tenfold, and changes no decision this year — the first reason
dominates. The argument for monitoring is for a book with more real effects in it.

In [ ]:
from axiom.design import tune

rho = tune(0.05, 1.0)
rows = []
for r in [b for b in book if b.metric == "primary"]:
    z = float(st.norm.isf(r.p_value / 2))
    rows.append([r.key.split("/")[1], f"{r.p_value:.4f}", f"{r.evalue:.2f}", f"{evalue(z, 1.0, rho=rho):.2f}",
                 "native" if r.experiment == "NW-01" else "calibrated"])
table(rows, headers=("experiment", "p", "e-value used", "mixture e-value at the final look", "which"),
      title="what a confidence sequence would have bought each readout")
print("NW-01's e-value used is the running maximum of its sequence, which is what Ville's inequality bounds; "
      "its final-look value is the smaller number in the fourth column.")

## 4 · The stream, decided on arrival

A quarterly review is the wrong shape for what actually happens: experiments finish
one at a time and somebody wants the answer that afternoon. `online_decisions` (LORD)
decides each readout as it lands and still controls the false discovery rate over the
whole stream, by giving each test a level out of a budget that is replenished every
time something is rejected. Arrival order is part of the procedure.

In [ ]:
from axiom.design import INDEPENDENT_READOUTS, online_decisions

finished = {k: reads[k].fielded.experiment.stop if k != "GX-02" else 36 for k in reads}
arrivals = sorted(book, key=lambda r: (finished[r.experiment], r.metric != "primary"))
stream = online_decisions(arrivals, alpha=0.05, stream="2026, as they landed")
print(stream.summary())
rows = []
for d in stream.decisions:
    r = arrivals[d.index - 1]
    rows.append([d.index, f"W{finished[r.experiment]:02d}", r.key.split("/")[1], f"{r.p_value:.4f}", f"{d.level:.4f}", "reject" if d.reject else "—"])
table(rows, headers=("arrival", "week", "readout", "p", "its own level", "decision"), title="the level each readout was judged at")
print("condition this route needs and the batch one does not:", INDEPENDENT_READOUTS.challenged_by[:120], "…")

In [ ]:
fig = steps(list(range(1, stream.n + 1)), list(stream.levels), color=VIOLET,
            title="The budget a stream spends and earns back",
            subtitle="the level each readout was judged at, in arrival order", x_title="arrival", y_title="alpha for this readout")
for d in stream.decisions:
    if d.reject:
        annotate(fig, d.index, d.level, arrivals[d.index - 1].key.split("/")[1], color=GOOD)
mark_y(fig, 0.05, text="the level each party would have used alone")
caption(fig, "Every readout is judged well below 0.05, and the level falls through each run of guardrails that "
             "return nothing; a rejection pays some of it back. A programme that discovers things can afford to keep "
             "testing. This one, mostly, cannot — and that is a finding about the book, not about the procedure.")

Three rejections, and one of them is the churn guardrail: it arrived directly after
GX-01 replenished the budget, at a level of 0.0116, with a p-value of 0.0109. The
procedure kept its promise — at most a twentieth of the rejections false, in
expectation, over a long stream — and this stream is short. The batch route, which
knew all twenty-four at once, did not take it.

## 5 · What each party is told

One table per party: the lift, the interval, what the party would have concluded on
its own, what the programme concludes, the run's conformance, and the departures a
reader of the number has to accept. The last two columns are the ones a board never
sees and should.

In [ ]:
decided = {d.key: d for d in ebh.decisions}
alone = {d.key: d for d in reports["none"].decisions}
for party in H.PARTIES:
    rows = []
    for r in [b for b in book if b.party == party and b.metric == "primary"]:
        rd = reads[r.experiment]
        m = rd.measurement
        v = rd.run.conformance()
        caveats = "; ".join(dv.role for dv in rd.run.deviations) or "—"
        rows.append([r.experiment, rd.plan.experiment.channel, f"{m.estimate:+.1f} ± {m.se:.1f}",
                     "go" if alone[r.key].go else "no", "go" if decided[r.key].go else "no", v.status, caveats])
    table(rows, headers=("experiment", "channel", "lift / market-week", "alone", "programme (e-BH)", "conformance", "departures"),
          title=f"{party}: what the house says")
quoted_alone = sum(1 for d in reports["none"].decisions if d.go and d.metric == "primary")
quoted = sum(1 for d in ebh.decisions if d.go and d.metric == "primary")
print(f"channels the house would have called incremental, party by party: {quoted_alone}; as a programme under e-BH: {quoted}")

## 6 · The audit trail

Six months from now somebody will ask what GX-02 was, exactly. The catalog's index has
the answer without opening a single artifact: every role the plan filed, the stopped
estimate, the measurement, and the run at every stage it passed through, all tagged
with the experiment id and held in globex's scope. And a builder for another party
cannot read it: asking for a digest that lives in someone else's scope is
`CrossScopeError`, not `None` and not the artifact.

In [ ]:
from axiom.io import CrossScopeError

trail = catalog.find(tags={"experiment": "GX-02"})
table([[e.type_name.split(":")[-1], e.label, e.tags.get("stage", "—"), e.digest[:12] + "…"] for e in trail],
      headers=("type", "label", "stage", "digest"), title="everything filed under GX-02, from the index alone")
by_type = pd.Series([e.type_name.split(":")[-1] for e in catalog.entries()]).value_counts()
print("the catalog, by type:", dict(by_type))
gx02_measurement = next(e for e in trail if e.type_name.endswith("Measurement"))
try:
    builders["acme"].store.get(gx02_measurement.digest)
except CrossScopeError as err:
    print("\nacme's builder asks for globex's measurement ->", type(err).__name__, "—", str(err)[:110])

## What notebook 4 established

1. **The block travels.** A run the lifecycle calls `blocked` never reaches the
   programme's error control; NW-03 was not excluded by a person remembering to.
2. **A book of twenty-four decisions expects more than one false go at 5%**, and got
   one on schedule: a guardrail nobody's channel touched. Corrected as a book, it goes
   away.
3. **Error control is not bias control.** The readout that survives the strictest
   correction is the one whose number notebook 3 showed is inflated by attrition. The
   warning is on the run, in the vocabulary every other verdict uses, and the
   multiplicity cannot see it.
4. **e-BH's bar is set by the book**: twenty-four readouts, one go, a cut of 480. The
   house monitored one experiment in nine, and monitoring the rest would have changed
   no decision this year; it is the argument for next year's dashboards being
   confidence sequences all the same.
5. **The stream can be decided on arrival**, at levels the budget can afford, and the
   budget this book earned back was small.
6. **The index answers the audit** without opening an artifact, and a party cannot read
   another party's.